**Ai Agent Customer Support**

Team members: << Fatimah Sultan Alzeer >>

This notebook builds a customer-support assistant that triages a ticket, retrieves answers from a knowledge base, hands off to the matching specialist agent, and escalates to a human when needed — using only the patterns taught in the course (agents & tools, sub-agents/handoffs, RAG, embeddings & semantic search, context & state, LangGraph Functional API, and LangSmith).

The single-paragraph write-up for every rubric section is at the end of the notebook.

## 0 · Setup

Add `OPENROUTER_API_KEY` and `LANGSMITH_API_KEY` to **Colab Secrets** (key icon in the
left sidebar). Get keys at https://openrouter.ai/keys and https://smith.langchain.com .

In [ ]:
%pip install -q langchain "langchain[openai]" langchain-community langchain-text-splitters \
    langchain-huggingface sentence-transformers langgraph

In [2]:
import os
from google.colab import userdata

# --- API keys ---
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

# --- LangSmith tracing (Rubric 8: observability) ---
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "capstone-customer-support"

In [3]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## 1 · Agent fundamentals — structured-output triage

A real LLM call that returns a **structured** triage decision (not a hardcoded output),
via `with_structured_output`. This is the router's brain.

In [4]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage, SystemMessage


class TicketClassification(BaseModel):
    """Structured triage decision for an incoming support ticket."""
    category: Literal["billing", "technical", "general", "complaint"] = Field(
        description="The domain the ticket belongs to."
    )
    urgency: Literal["low", "medium", "high", "critical"] = Field(
        description="How urgent the ticket is."
    )
    summary: str = Field(description="One-sentence summary of the customer's issue.")


classifier = model_nemotron3_nano.with_structured_output(TicketClassification)


def classify(ticket_text: str) -> TicketClassification:
    return classifier.invoke([
        SystemMessage("You triage customer support tickets. Classify the ticket."),
        HumanMessage(ticket_text),
    ])

## 2 · RAG pipeline — load → split → embed → store → retrieve

I build a support knowledge base the exact way the course does it: each article is a
`Document`, split with `RecursiveCharacterTextSplitter` (1000 / 200 overlap), embedded
with HuggingFace `all-mpnet-base-v2`, stored in an `InMemoryVectorStore`, and queried
through `similarity_search` / `as_retriever`.

*(Swap the in-code `Document` list for `PyPDFLoader`/`WebBaseLoader` to ingest real
manuals — the rest of the pipeline is identical.)*

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 1) Load
kb_articles = [
    Document(page_content=(
        "To reset your password, go to Settings > Security > Change Password. A reset "
        "link is emailed to the address on file and expires after 30 minutes."),
        metadata={"source": "kb", "topic": "account"}),
    Document(page_content=(
        "Refunds are issued to the original payment method within 5-7 business days. "
        "Duplicate charges are auto-detected and reversed within 3 business days."),
        metadata={"source": "kb", "topic": "billing"}),
    Document(page_content=(
        "If the app crashes on launch, update to the latest version, clear the cache, "
        "and restart the device. Crashes that persist after updating are logged as bugs."),
        metadata={"source": "kb", "topic": "technical"}),
    Document(page_content=(
        "Hardware under warranty is repaired free of charge at authorized service "
        "centers. Accidental damage may be excluded and quoted as a paid repair."),
        metadata={"source": "kb", "topic": "warranty"}),
    Document(page_content=(
        "Subscriptions renew automatically. Cancel anytime from Settings > Billing > "
        "Manage Subscription; access continues until the end of the paid period."),
        metadata={"source": "kb", "topic": "billing"}),
]

# 2) Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(kb_articles)

# 3) Embed  +  4) Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(all_splits)

# 5) Retrieve
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print(f"Indexed {len(all_splits)} chunks.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 5 chunks.


**Hybrid RAG.** I use *both* formulations the course teaches:
* a fast **2-step** retrieval (`retrieve_context`) to pre-load context for triage/drafting, and
* an **agentic** search **tool** (`search_knowledge_base`) that the specialist agent calls at its own discretion for multi-step technical questions.

Justification is in the write-up (Section 3).

In [6]:
from langchain.tools import tool


def retrieve_context(query: str) -> str:
    """2-step retrieval: pull the top-k KB passages for a query (pre-loaded context)."""
    docs = vector_store.similarity_search(query, k=2)
    return "\n\n".join(f"Source: {d.metadata}\nContent: {d.page_content}" for d in docs)


@tool
def search_knowledge_base(query: str) -> str:
    """Search the support knowledge base for passages relevant to the query."""
    return retrieve_context(query)

## 3 · Context & State — short-term vs. long-term memory

* **Runtime context** (`SupportContext`) — immutable per-session identity (`customer_id`).
* **Short-term memory** — the per-thread conversation, persisted by a **checkpointer**
  keyed on `thread_id` (this is what lets the workflow pause on an `interrupt()` and resume).
* **Long-term memory** (`CUSTOMER_MEMORY`) — a **cross-session** store of the customer's
  profile and past tickets. This is *not* the chat-history list: it survives across
  different threads. In production this is a database; here an in-memory dict.

In [7]:
from dataclasses import dataclass
from typing_extensions import NotRequired
from langchain.agents import AgentState

# Runtime context: immutable per-session identity
@dataclass
class SupportContext:
    customer_id: str

# Short-term memory schema (extends the built-in messages state)
class SupportState(AgentState):
    category: NotRequired[str]
    urgency: NotRequired[str]

# Long-term memory: cross-session store keyed by customer_id
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust_001": {"name": "Alice Johnson", "plan": "Premium", "past_tickets": []},
}

def load_customer_memory(customer_id: str) -> dict:
    return CUSTOMER_MEMORY.setdefault(
        customer_id, {"name": "Unknown", "plan": "Standard", "past_tickets": []}
    )

def save_ticket_to_memory(customer_id: str, summary: str) -> None:
    load_customer_memory(customer_id)["past_tickets"].append(summary)

In [8]:
from langchain.tools import ToolRuntime

@tool
def get_customer_profile(runtime: ToolRuntime[SupportContext]) -> str:
    """Get the current customer's profile and ticket history (long-term memory)."""
    profile = load_customer_memory(runtime.context.customer_id)
    past = "; ".join(profile["past_tickets"]) or "none"
    return f"Name: {profile['name']}\nPlan: {profile['plan']}\nPast tickets: {past}"

## 4 · Multi-agent / handoff architecture — specialist agents

Two specialist agents built with `create_agent` (real tool calls). The workflow **hands
off** each ticket to the one matching its category. The technical agent does **agentic
RAG**; both read **long-term memory** through `get_customer_profile` (runtime context).
`SummarizationMiddleware` keeps short-term memory bounded on long chats.

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

@tool
def lookup_order(order_id: str) -> str:
    """Look up the status of a billing order by its ID."""
    orders = {
        "A-100": "charged twice on 2026-01-03; duplicate auto-reversal pending",
        "A-200": "paid, active subscription",
    }
    return orders.get(order_id, "Order not found. Ask the customer to confirm the order ID.")


technical_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[search_knowledge_base, get_customer_profile],
    context_schema=SupportContext,
    state_schema=SupportState,
    system_prompt=("You are a technical support specialist. Use search_knowledge_base to "
                   "ground every answer in the manuals before responding."),
    middleware=[SummarizationMiddleware(
        model=model_nemotron3_nano, trigger=("tokens", 4000), keep=("messages", 10))],
)

billing_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[lookup_order, search_knowledge_base, get_customer_profile],
    context_schema=SupportContext,
    state_schema=SupportState,
    system_prompt=("You are a billing specialist. Look up orders and KB policy before "
                   "answering billing questions."),
)

## 5 · LangGraph Functional API + error handling

Each step is a `@task`; the `@entrypoint` is the orchestrator. I implement **all four**
course error-handling strategies:

1. **Transient → automatic retry** with `RetryPolicy` on the classify task.
2. **LLM-recoverable → loop back** — catch an empty specialist reply and re-prompt.
3. **User-fixable → `interrupt()`** — pause to collect a missing order ID.
4. **Unexpected → bubble up** — let real failures surface in `send_reply`.

In [10]:
from langgraph.func import task, entrypoint
from langgraph.types import interrupt, RetryPolicy, Command
from langgraph.checkpoint.memory import InMemorySaver


# (1) Transient errors -> automatic retry
@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0))
def classify_ticket(ticket_text: str) -> dict:
    """Classify the ticket. Retried automatically on transient API/network failures."""
    return classify(ticket_text).model_dump()


# (2) LLM-recoverable -> catch a bad/empty reply and loop back
@task
def run_specialist(category: str, ticket_text: str, context: str, customer_id: str) -> str:
    """Hand off to the matching specialist agent (billing/technical) and return its reply."""
    agent = billing_agent if category == "billing" else technical_agent
    prompt = f"Customer ticket: {ticket_text}\n\nKB context:\n{context}"
    reply = ""
    for _ in range(2):
        result = agent.invoke(
            {"messages": [HumanMessage(prompt)]},
            context=SupportContext(customer_id=customer_id),
        )
        reply = result["messages"][-1].content
        if reply and reply.strip():
            return reply
        # loop back: let the agent see it failed and try again
        prompt += "\n\n(Your previous answer was empty. Answer using your tools.)"
    return reply


# (3) User-fixable -> interrupt to collect a missing order id
@task
def ensure_order_id(order_id) -> str:
    """Billing needs an order id; if missing, pause and ask the human customer for it."""
    if not order_id:
        provided = interrupt({"message": "Order ID needed",
                              "request": "Please provide your order ID (e.g. A-100)."})
        order_id = provided["order_id"] if isinstance(provided, dict) else provided
    return order_id


# Human-in-the-loop handoff: pause for a human agent to approve or take over
@task
def escalate_to_human(draft: str, classification: dict) -> dict:
    """Pause the workflow and hand off to a human agent for approval / takeover."""
    return interrupt({
        "action": "Human approval required before sending",
        "draft_response": draft,
        "category": classification["category"],
        "urgency": classification["urgency"],
    })


# (4) Unexpected -> let it bubble up for debugging
@task
def send_reply(reply: str) -> str:
    """Send the reply to the customer (mocked). Unexpected errors bubble up."""
    try:
        # e.g. email_service.send(reply)
        print(f"[SENT to customer] {reply[:80]}...")
        return "sent"
    except Exception:
        raise  # surface unexpected errors to the developer

## 6 · The workflow — Routing + Prompt Chaining

**Workflow pattern:** *Routing* (classify, then dispatch to a specialist) composed with
*Prompt Chaining* (classify → retrieve → draft → review → send). The `checkpointer` +
`thread_id` make the whole thing durable across every `interrupt()`.

In [11]:
checkpointer = InMemorySaver()

@entrypoint(checkpointer=checkpointer)
def support_workflow(inputs: dict) -> dict:
    """Triage -> retrieve -> handoff to specialist -> (maybe) human handoff -> send."""
    ticket_text = inputs["ticket_text"]
    customer_id = inputs["customer_id"]
    order_id = inputs.get("order_id")

    # Long-term memory: load the customer's profile/history (cross-session)
    profile = load_customer_memory(customer_id)

    # Step 1 - triage (structured output; transient-retry task)  [ROUTING]
    classification = classify_ticket(ticket_text).result()

    # Step 2 - hybrid RAG: fast 2-step retrieval to pre-load context  [PROMPT CHAINING]
    context = retrieve_context(classification["summary"])

    # Step 3 - billing may need a user-fixable order id (interrupt if missing)
    if classification["category"] == "billing":
        order_id = ensure_order_id(order_id).result()
        context += f"\n\nOrder {order_id}: {lookup_order.invoke({'order_id': order_id})}"

    # Step 4 - HANDOFF to the matching specialist agent (agentic RAG)
    draft = run_specialist(
        classification["category"], ticket_text, context, customer_id
    ).result()

    # Step 5 - human-in-the-loop: escalate complaints / high-urgency before sending
    needs_human = (classification["urgency"] in ("high", "critical")
                   or classification["category"] == "complaint")
    if needs_human:
        decision = escalate_to_human(draft, classification).result()
        if not decision.get("approved", False):
            return {"status": "handed_off_to_human", "final_response": None,
                    "classification": classification}
        draft = decision.get("edited_response", draft)

    # Step 6 - send + persist to long-term memory
    send_reply(draft).result()
    save_ticket_to_memory(customer_id, classification["summary"])
    return {"status": "sent", "final_response": draft,
            "classification": classification, "profile": profile}

## 7 · Run it — with two real human-in-the-loop pauses

A high-urgency billing complaint about a double charge. It has no order ID, so the
workflow **interrupts** to ask for one; after resuming, high urgency triggers a second
**interrupt** for human approval before the reply is sent.

In [12]:
import uuid

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

inputs = {
    "ticket_text": "I was charged twice for my subscription and I'm furious! Fix it now.",
    "customer_id": "cust_001",
}

result = support_workflow.invoke(inputs, config)
print("PAUSE 1 (needs order id):")
print(result["__interrupt__"])

PAUSE 1 (needs order id):
[Interrupt(value={'message': 'Order ID needed', 'request': 'Please provide your order ID (e.g. A-100).'}, id='2a8997b24c5f584b194f9a83adfbdb63')]


In [13]:
# Resume #1: provide the missing order id (user-fixable interrupt resolved)
result = support_workflow.invoke(Command(resume={"order_id": "A-100"}), config)
print("PAUSE 2 (human approval):")
print(result["__interrupt__"])

PAUSE 2 (human approval):
[Interrupt(value={'action': 'Human approval required before sending', 'draft_response': 'I’m really sorry you’ve been double‑charged – that’s definitely not the experience we want you to have.  \n\n**What’s happening with Order\u202fA‑100:**  \n- Our system has already flagged the duplicate charge and an **auto‑reversal is pending**.  \n- The reversal will be completed within **3 business days**.  \n- Once the reversal posts, any resulting refund will be returned to the original payment method and should appear on your statement within **5‑7 business days**.\n\nIn short, the duplicate charge is being taken care of automatically, and you’ll see the correction on your account shortly.  \n\nIf you don’t see the reversal reflected after a few days, or if you have any other concerns (e.g., wanting to cancel the subscription, update payment details, or need a direct follow‑up), just let me know and I’ll gladly help you right away.  \n\nAgain, I apologize for the inc

In [14]:
# Resume #2: the human approves / edits the reply -> workflow finishes
result = support_workflow.invoke(
    Command(resume={"approved": True,
                    "edited_response": ("We've reversed the duplicate charge; your refund "
                                        "lands within 3 business days. Sincere apologies.")}),
    config,
)
print(result["status"])
print(result["final_response"])

[SENT to customer] We've reversed the duplicate charge; your refund lands within 3 business days. S...
sent
We've reversed the duplicate charge; your refund lands within 3 business days. Sincere apologies.


In [15]:
# Long-term memory persisted the resolved ticket across the session:
print(CUSTOMER_MEMORY["cust_001"]["past_tickets"])

['Duplicate charge on subscription']


## 8 · Write-up

**Agent fundamentals.** The triage brain is a real `nvidia/nemotron-3-nano` call wrapped
with `with_structured_output(TicketClassification)`, so every ticket yields a typed
`category / urgency / summary` object rather than a hardcoded string. The specialist
agents (`create_agent`) then make genuine tool calls (`lookup_order`,
`search_knowledge_base`, `get_customer_profile`). Structured output is used where a
machine-readable decision is needed (routing), and free-form generation where a human
reply is needed (the specialist's answer).

**Multi-agent / handoff architecture.** I follow the course's handoff pattern: a triage
step routes each ticket and **hands it off** to the matching specialist agent, and any
high-urgency or complaint ticket is **handed off to a human** through the
`escalate_to_human` + `interrupt()` mechanism. This is a deliberate supervisor-style
routing design, not an ad-hoc one — the entrypoint is the supervisor and the specialists
are the workers.

**RAG pipeline.** A full retrieval pipeline runs: `Document`s are **loaded**, **split**
(`RecursiveCharacterTextSplitter`, 1000/200), **embedded** (`all-mpnet-base-v2`),
**stored** (`InMemoryVectorStore`), and **retrieved** (`similarity_search` /
`as_retriever`). I chose **Hybrid RAG**: a cheap 2-step retrieval pre-loads context for
triage/drafting (fast, predictable, one inference call), while the technical specialist
also holds an **agentic** `search_knowledge_base` tool so it can issue extra, contextual
searches on multi-step questions. Pure 2-step is too rigid for open technical queries;
pure agentic wastes an inference call on trivial tickets — hybrid gets both.

**Context & state management.** The `@entrypoint` runs on an `InMemorySaver` checkpointer
keyed by `thread_id`, which is what makes the two `interrupt()`/resume round-trips
durable — that is **short-term** memory. Separately, `CUSTOMER_MEMORY` is **long-term**
memory: a cross-session store of the customer's profile and past tickets, read via the
`get_customer_profile` tool (runtime context) and written at the end of each resolution.
It is explicitly not the chat-history list. `SummarizationMiddleware` bounds the
specialist's short-term history on long conversations.

**Human-in-the-loop.** There are two real `interrupt()` pause points: a *user-fixable*
pause that collects a missing order ID, and a *handoff* pause that requires a human agent
to approve or edit the draft before it is sent. Each resumes with `Command(resume=...)`
on the same `thread_id`, and the human's edited text becomes the final reply.

**LangGraph Functional API & error handling.** Every step is a `@task` orchestrated by an
`@entrypoint` with plain Python control flow and `.result()`. All four error-handling
strategies are implemented: **transient retry** (`RetryPolicy` on `classify_ticket`),
**LLM-recoverable loopback** (empty specialist reply is caught and re-prompted),
**user-fixable interrupt** (`ensure_order_id`), and **unexpected bubble-up** (`send_reply`
re-raises).

**Workflow pattern.** The system explicitly implements **Routing** (a classifier chooses
the branch/specialist) composed with **Prompt Chaining** (classify → retrieve → draft →
review → send). Both are named course patterns.

**LangSmith observability.** Tracing is enabled via LANGSMITH_TRACING / LANGSMITH_API_KEY / LANGSMITH_PROJECT. Inspecting a run of the demo ticket in the capstone-customer-support project, the trace showed the run_specialist step (the billing agent) as the latency bottleneck because it issued two sequential tool calls — lookup_order then search_knowledge_base — whereas classify_ticket returned on its first attempt with no retry, confirming the structured-output triage call was stable. The two interrupt() pauses also appear in the trace as separate resumed runs sharing a single thread_id.

In [ ]:
import json
from google.colab import _message, files

nb = _message.blocking_request('get_ipynb', timeout_sec=60)['ipynb']

nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    if isinstance(cell.get("metadata"), dict):
        cell["metadata"].pop("widgets", None)
    for out in cell.get("outputs", []):
        if isinstance(out.get("metadata"), dict):
            out["metadata"].pop("widgets", None)
        data = out.get("data", {})
        if isinstance(data, dict):
            data.pop("application/vnd.jupyter.widget-view+json", None)

filename = "Ai_agent.ipynb"

with open(filename, "w") as f:
    json.dump(nb, f, indent=1)

files.download(filename)
